# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadQasimTahir/flyrank_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


Unit of Analysis: One row represents one pseudonymized content item (content_hash_id) for a specific client (client_hash_id) aggregated over a monthly period.  

Time Window: March 2026 (2026-03-01 to 2026-03-31). We deliberately use a mid-panel month to build and test our logic, leaving the final available month (June 2026) untouched as a sealed holdout set for future validation.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


Features (The 5 safe signals):

impressions_total: Knowable at the decision moment because it aggregates search visibility up to the present day.

clicks_total: Knowable at the decision moment because it reflects actual realized search traffic.

avg_position_mean: Knowable at the decision moment because it tracks historical SERP placement.

days_active: Knowable at the decision moment because the duration a page has been tracked is a historical fact.

historical_ctr: Knowable at the decision moment because it is derived entirely from past clicks and impressions.

Label (Proxy): is_declining (A binary target: 1 if impressions in the last 7 days of March were lower than the first 7 days, 0 otherwise).

Context: content_hash_id.

Excluded: LEAKED_imp_diff (the exact mathematical difference between the first 7 days and last 7 days of impressions). This must be excluded because the label is directly derived from it; including it would create a circular loop and guarantee feature leakage.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

# 1. Setting up Hugging Face Auth & DuckDB Connection
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = 'YOUR_HF_TOKEN_HERE'

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

base_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

# Query 1: Row count & Date span for March 2026
print("--- 1. Row Count & Date Span ---")
span_query = f"""
SELECT
    COUNT(*) as total_rows,
    MIN(report_date) as start_date,
    MAX(report_date) as end_date
FROM '{base_path}'
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
"""
display(con.execute(span_query).df())

# Query 2: Grain Check (Aggregating daily facts to one row per content item)
print("\n--- 2. Grain Check (One row per content item) ---")
grain_query = f"""
SELECT
    content_hash_id,
    COUNT(*) as daily_records_found
FROM '{base_path}'
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
GROUP BY content_hash_id
LIMIT 5
"""
display(con.execute(grain_query).df())

# Query 3: Availability Check (IS TRUE)
print("\n--- 3. Availability Check (ga4_data_available) ---")
avail_query = f"""
SELECT COUNT(*) as rows_with_ga4_active
FROM '{base_path}'
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
  AND ga4_data_available IS TRUE
"""
display(con.execute(avail_query).df())

# 4. The 5-Feature Frame & The Leakage Trap
print("\n--- 4. Feature Frame & The Leakage Trap ---")
feature_query = f"""
WITH march_agg AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) as impressions_total,
        SUM(gsc_clicks) as clicks_total,
        AVG(gsc_avg_position) as avg_position_mean,
        MAX(report_date) - MIN(report_date) as days_active,
        SUM(CASE WHEN report_date >= '2026-03-25' THEN gsc_impressions ELSE 0 END) as last_7d_imp,
        SUM(CASE WHEN report_date <= '2026-03-07' THEN gsc_impressions ELSE 0 END) as first_7d_imp
    FROM '{base_path}'
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
    GROUP BY content_hash_id
)
SELECT
    impressions_total,
    clicks_total,
    avg_position_mean,
    CAST(days_active AS INT) as days_active,
    (clicks_total / NULLIF(impressions_total, 0)) as historical_ctr,
    CASE WHEN last_7d_imp < first_7d_imp THEN 1 ELSE 0 END as is_declining,
    (last_7d_imp - first_7d_imp) as LEAKED_imp_diff
FROM march_agg
WHERE impressions_total > 100
LIMIT 5000
"""
df_features = con.execute(feature_query).df().fillna(0)

# Springing the Trap
features = ["impressions_total", "clicks_total", "avg_position_mean", "days_active", "historical_ctr"]
X_leaked = df_features[features + ["LEAKED_imp_diff"]]
y = df_features["is_declining"]

trap_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaked, y)
print(f"Score WITH leaked feature (Precision): {precision_score(y, trap_model.predict(X_leaked)):.3f} <- The trap (perfect/near-perfect score)")

# Removing the leak for the honest baseline
X_honest = df_features[features]
honest_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_honest, y)
print(f"Score WITHOUT leaked feature (Precision): {precision_score(y, honest_model.predict(X_honest)):.3f} <- The honest baseline")

--- 1. Row Count & Date Span ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31



--- 2. Grain Check (One row per content item) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_with_ga4_active
0,413966



--- 4. Feature Frame & The Leakage Trap ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Score WITH leaked feature (Precision): 1.000 <- The trap (perfect/near-perfect score)
Score WITHOUT leaked feature (Precision): 0.591 <- The honest baseline


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation of this slice: The historical data represents an unbalanced panel. Because clients onboarded tracking at different times (visible via ga4_data_start), querying older windows means some pages might artificially appear to have zero traffic. This absence of data indicates tracking was inactive, not necessarily that the content failed to perform.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.